# API 14 - Contrato público (CLI)

Notebook CLI paralelo a `tutorials/api/14_api_contract_matrix.ipynb`.

**Objetivo:** Listar y ejercitar el inventario de contratos públicos.

Este notebook no llama factories de Agentic Systems directamente: ejecuta el
entrypoint CLI real, conserva la salida Rich y valida después el JSON del mismo
contrato.


## Cómo se ejecuta

La forma portable es `python -m agentic_systems.cli ...`. Después de instalar
el wheel, el entrypoint equivalente es `agentic-systems ...`.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def _repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio.")


ROOT = _repo_root()
CLI = [sys.executable, "-m", "agentic_systems.cli"]


def run_cli(*args: str, expected: int = 0) -> str:
    env = os.environ.copy()
    source_path = str(ROOT / "src")
    env["PYTHONPATH"] = (
        source_path
        if not env.get("PYTHONPATH")
        else source_path + os.pathsep + env["PYTHONPATH"]
    )
    command = [*CLI, *args]
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )
    print("$ " + " ".join(command))
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    assert completed.returncode == expected, completed.stderr
    return completed.stdout


def run_cli_json(*args: str) -> dict:
    return json.loads(run_cli(*args, "--json"))


def assert_rich(output: str, title: str) -> None:
    assert title in output
    ascii_box = "+" in output and "|" in output
    unicode_box = "─" in output and "│" in output
    assert ascii_box or unicode_box


## 1) Salida humana Rich

La celda conserva stdout y comprueba título y bordes. Esto detecta tablas o
paneles truncados, además del exit code.


In [2]:
rich_output = run_cli(*['api', 'list', '--tier', 'public'])
assert_rich(rich_output, 'API Inventory')


$ C:\Users\jacob\Documents\AVANTECK.TEAM\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\.venv\Scripts\python.exe -m agentic_systems.cli api list --tier public
+------------------------------- API Inventory -------------------------------+
| tier: public                                                                |
| count: 373                                                                  |
+-----------------------------------------------------------------------------+
skill                                                                          
agent                                                                          
system                                                                         
environment                                                                    
eval                                                                           
runtime                                                                        
provider                        

## 2) Contrato de máquina

La misma ruta se ejecuta con `--json` para afirmar campos y cardinalidad sin
parsear la presentación Rich.


In [3]:
payload = run_cli_json(*['api', 'list', '--tier', 'public'])
assert payload["count"] >= 300
payload


$ C:\Users\jacob\Documents\AVANTECK.TEAM\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\.venv\Scripts\python.exe -m agentic_systems.cli api list --tier public --json
{
  "count": 373,
  "ids": [
    "skill",
    "agent",
    "system",
    "environment",
    "eval",
    "runtime",
    "provider",
    "framework",
    "scheduler",
    "load_skill",
    "AgenticSystem",
    "AgenticSystem.add",
    "AgenticSystem.agent",
    "AgenticSystem.agents",
    "AgenticSystem.arun",
    "AgenticSystem.compile",
    "AgenticSystem.composition",
    "AgenticSystem.environment",
    "AgenticSystem.eval",
    "AgenticSystem.execute_tool",
    "AgenticSystem.export_tool_specs",
    "AgenticSystem.graph",
    "AgenticSystem.inspect",
    "AgenticSystem.load_skill",
    "AgenticSystem.public_tool_names",
    "AgenticSystem.public_tools",
    "AgenticSystem.run",
    "AgenticSystem.runtime_skills",
    "AgenticSystem.skill",
    "AgenticSystem.skill_names",
    "AgenticSystem.skills",
    "Agenti

{'count': 373,
 'ids': ['skill',
  'agent',
  'system',
  'environment',
  'eval',
  'runtime',
  'provider',
  'framework',
  'scheduler',
  'load_skill',
  'AgenticSystem',
  'AgenticSystem.add',
  'AgenticSystem.agent',
  'AgenticSystem.agents',
  'AgenticSystem.arun',
  'AgenticSystem.compile',
  'AgenticSystem.composition',
  'AgenticSystem.environment',
  'AgenticSystem.eval',
  'AgenticSystem.execute_tool',
  'AgenticSystem.export_tool_specs',
  'AgenticSystem.graph',
  'AgenticSystem.inspect',
  'AgenticSystem.load_skill',
  'AgenticSystem.public_tool_names',
  'AgenticSystem.public_tools',
  'AgenticSystem.run',
  'AgenticSystem.runtime_skills',
  'AgenticSystem.skill',
  'AgenticSystem.skill_names',
  'AgenticSystem.skills',
  'AgenticSystem.tool',
  'AgenticSystem.tool_names',
  'AgenticSystem.toolkit',
  'AgenticSystem.tools',
  'AgenticSystem.toolset',
  'Agent',
  'Agent.arun',
  'Agent.as_async_node',
  'Agent.as_node',
  'Agent.as_tool',
  'Agent.available_tools',
  'Ag

## Resultado e interpretación

Rich responde a lectura humana; JSON responde a automatización. Ambos nacen del
mismo comando y del mismo escenario público. Un estado `not-run` conserva el
motivo, pero no cuenta como evidencia live.
